# Kasus A: Deteksi Penipuan Pembukaan Rekening (BAF Base)

Target metrik: **TPR pada FPR 5%** (persentase penipu tertangkap saat nasabah jujur yang salah tertandai maksimal 5%).

Data: `data/Base.csv` dari Kaggle *Bank Account Fraud Dataset (NeurIPS 2022)*, hanya baris `month` 0-3.

In [1]:
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "Base.csv"
MODEL_PATH = ROOT / "models" / "kasus_a_fraud_lgbm.joblib"
SEED = 42

## 1. Muat data dan filter bulan 0-3

In [2]:
df = pd.read_csv(DATA)
df = df[df["month"].between(0, 3)].reset_index(drop=True)
print(df.shape)
df.groupby("month")["fraud_bool"].agg(["size", "mean"])

(547975, 32)


,size,mean
month,,
0,132440,0.011326
1,127620,0.009387
2,136979,0.008746
3,150936,0.009222


## 2. Fitur

- `device_fraud_count` hanya punya satu nilai, jadi dibuang.
- `month` dipakai untuk membagi data, bukan sebagai fitur (model harus berlaku untuk bulan berikutnya).
- Kolom teks dijadikan tipe `category` agar LightGBM menanganinya langsung.
- Nilai `-1` pada beberapa kolom berarti data kosong; LightGBM bisa memisahkannya sendiri, jadi dibiarkan.

In [3]:
TARGET = "fraud_bool"
DROP = [TARGET, "month", "device_fraud_count"]
CAT_COLS = df.select_dtypes("object").columns.tolist()
FEATURES = [c for c in df.columns if c not in DROP]

categories = {c: sorted(df[c].unique()) for c in CAT_COLS}


def prepare(frame: pd.DataFrame) -> pd.DataFrame:
    X = frame[FEATURES].copy()
    for c in CAT_COLS:
        X[c] = pd.Categorical(X[c], categories=categories[c])
    return X


print(len(FEATURES), "fitur;", "kategorikal:", CAT_COLS)

29 fitur; kategorikal: ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']


## 3. Pembagian data: berdasarkan waktu

- **Latih**: bulan 0-2, **Validasi**: bulan 3.
- Alasan: di produksi model dilatih dengan data lama lalu menilai pengajuan yang datang *sesudahnya*. Split acak akan mencampur bulan yang sama di latih dan validasi sehingga skor terlalu optimis dan tidak menangkap pergeseran pola penipuan antar bulan.
- Untuk memilih jumlah pohon (early stopping) dipakai bulan 2 sebagai holdout di dalam data latih, supaya bulan 3 benar-benar tidak tersentuh sampai evaluasi akhir.

In [4]:
train = df[df["month"] <= 2]
valid = df[df["month"] == 3]
X_train, y_train = prepare(train), train[TARGET]
X_valid, y_valid = prepare(valid), valid[TARGET]
print("latih:", X_train.shape, f"fraud {y_train.mean():.4f}")
print("validasi:", X_valid.shape, f"fraud {y_valid.mean():.4f}")

latih: (397039, 29) fraud 0.0098
validasi: (150936, 29) fraud 0.0092


In [5]:
def tpr_at_fpr(y_true, scores, max_fpr=0.05):
    fpr, tpr, thr = roc_curve(y_true, scores)
    idx = np.searchsorted(fpr, max_fpr, side="right") - 1
    return tpr[idx], thr[idx]

## 4. Model: LightGBM

Alasan memilih LightGBM:
- Data tabular campuran numerik dan kategorikal: gradient boosting tree biasanya terbaik untuk jenis data ini, dan juga baseline terkuat di paper BAF.
- Cepat dilatih di CPU untuk ratusan ribu baris, tidak butuh scaling, menangani kategori dan nilai kosong secara bawaan.
- Penipuan hanya ~1% data; `scale_pos_weight` menaikkan bobot kelas penipu. Metrik TPR@FPR hanya bergantung pada urutan skor, jadi bobot ini tidak merusak evaluasi.

In [6]:
PARAMS = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=200,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_lambda=5.0,
    scale_pos_weight=10.0,
    n_estimators=3000,
    random_state=SEED,
    verbose=-1,
)

inner_tr = train["month"] <= 1
es_model = lgb.LGBMClassifier(**PARAMS)
es_model.fit(
    X_train[inner_tr.values], y_train[inner_tr.values],
    eval_set=[(X_train[~inner_tr.values], y_train[~inner_tr.values])],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(200, verbose=False)],
)
best_iter = es_model.best_iteration_
print("jumlah pohon terbaik:", best_iter)

jumlah pohon terbaik: 445


In [7]:
model = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": best_iter})
model.fit(X_train, y_train)

valid_scores = model.predict_proba(X_valid)[:, 1]
tpr, thr = tpr_at_fpr(y_valid, valid_scores)
print(f"Validasi (bulan 3) ROC-AUC     : {roc_auc_score(y_valid, valid_scores):.4f}")
print(f"Validasi (bulan 3) TPR @ FPR 5%: {tpr:.4f}  (threshold skor {thr:.4f})")

Validasi (bulan 3) ROC-AUC     : 0.9124
Validasi (bulan 3) TPR @ FPR 5%: 0.5819  (threshold skor 0.3674)


## 5. Fitur paling berpengaruh

In [8]:
pd.Series(model.booster_.feature_importance("gain"), index=FEATURES).sort_values(ascending=False).head(10)

housing_status                  348671.521843
device_os                       170615.947926
current_address_months_count    105415.139298
name_email_similarity            99008.642317
credit_risk_score                92030.932642
intended_balcon_amount           69974.786757
has_other_cards                  67724.198301
bank_branch_count_8w             66415.711636
velocity_4w                      66264.026730
days_since_request               66066.300606
dtype: float64

## 6. Simpan model lalu muat ulang

Disimpan bersama daftar fitur, kategori, dan threshold pada FPR 5% agar bisa langsung dipakai untuk skor pengajuan baru.

In [9]:
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(
    {"model": model, "features": FEATURES, "categories": categories, "threshold_fpr5": float(thr)},
    MODEL_PATH,
)

bundle = joblib.load(MODEL_PATH)
reloaded_scores = bundle["model"].predict_proba(prepare(valid))[:, 1]
assert np.allclose(reloaded_scores, valid_scores)
print("Model dimuat ulang dari", MODEL_PATH.relative_to(ROOT))
print(f"TPR @ FPR 5% dari model yang dimuat ulang: {tpr_at_fpr(y_valid, reloaded_scores)[0]:.4f}")

Model dimuat ulang dari models/kasus_a_fraud_lgbm.joblib
TPR @ FPR 5% dari model yang dimuat ulang: 0.5819
